# Format for SFT  (Prompt & Dataset Formatting)

This notebook converts the filtered **Islamic Services** subset into **Gemma 3 / AISA Function Calling** training format.

## Inputs
- `aisa_islamic_train` — training data (from Member 1)
- `aisa_islamic_dev` — development data

## Outputs (`formatted/` folder)
| File | Description |
|------|-------------|
| `islamic_train_track_b.jsonl` | Training with `<think>` + tool call |
| `islamic_train_track_a.jsonl` | Training with tool call only (no reasoning) |
| `islamic_dev_eval.jsonl` | Prompts + gold labels for evaluation |

> **Track B** = Arabic reasoning trace, then function call  
> **Track A** = function call only

## 0. Install dependencies (run once)

Run the cell below first. It installs into **the same Python kernel** this notebook uses, then print the Python path so you can confirm the environment.

In [2]:
import sys
import subprocess

def ensure_package(package: str) -> None:
    try:
        __import__(package)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

ensure_package("datasets")
ensure_package("pandas")

print("Python:", sys.executable)

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device



## 1. Setup — Imports & Constants

Import libraries and define constants used in the Gemma 3 format.

In [5]:
import json
import re
from collections import Counter
from pathlib import Path
from typing import Any

from datasets import Dataset, load_dataset, load_from_disk

# Gemma 3 model-turn marker
MODEL_TURN = "<start_of_turn>model\n"
END_CALL = "<end_function_call>"

# Used to strip the think block for Track A
THINK_BLOCK_RE = re.compile(
    r"<think>\s*.*?\s*</think>\s*",
    re.DOTALL,
)

ISLAMIC_TOOLS = {
    "get_qibla_direction",
    "calculate_zakat",
    "search_quran",
    "calculate_inheritance",
}

# Paths
ROOT = Path(".").resolve()
TRAIN_DIR = ROOT / "aisa_islamic_train"
DEV_DIR = ROOT / "aisa_islamic_dev"
OUTPUT_DIR = ROOT / "formatted"
PREVIEW_COUNT = 3

ModuleNotFoundError: No module named 'datasets'

## 2. Load Data

Read the datasets saved by Member 1. If they are missing, download from HuggingFace and filter.

In [ ]:
def load_split(path: Path, split_name: str) -> Dataset:
    if path.exists():
        return load_from_disk(str(path))

    dataset = load_dataset("TuwaiqAcademy/AISA-ArabicFC")
    split = dataset["train" if split_name == "train" else "dev"]
    return split.filter(lambda example: example["tool_called"] in ISLAMIC_TOOLS)


train_ds = load_split(TRAIN_DIR, "train")
dev_ds = load_split(DEV_DIR, "dev")

print(f"Train: {len(train_ds)} examples")
print(f"Dev:   {len(dev_ds)} examples")
print(f"Features: {list(train_ds.features.keys())}")

## 3. Explore One Example

Each row has a ready-made `text` field in Gemma format. We split it into:
- **prompt** → model input (system + tools + user)
- **completion** → what the model should learn (think + function call)

In [ ]:
example = train_ds[0]

print("Tool called:", example["tool_called"])
print("Dialect:", example["dialect"])
print("Requires function:", example["requires_function"])
print("Tools sampled:", [t["function"]["name"] for t in example["tools_sampled"]])

# User message from messages
for msg in example["messages"]:
    if msg["role"] == "user":
        print("\nUser:", msg["content"])

In [ ]:
# Preview the text field — model output portion
text = example["text"]
_, raw_completion = text.split(MODEL_TURN, 1)

print("=== Completion (model should learn this) ===")
print(raw_completion[:600])
print("...")
print("\nHas <start_function_response>:", "<start_function_response>" in raw_completion)

## 4. Helper Functions

### 4.1 Extract fields from `messages`

In [ ]:
def clean_arguments(arguments: dict[str, Any] | None) -> dict[str, Any]:
    """Remove null values from arguments."""
    if not arguments:
        return {}
    return {key: value for key, value in arguments.items() if value is not None}


def get_user_message(example: dict[str, Any]) -> str:
    for message in example["messages"]:
        if message["role"] == "user":
            return message.get("content", "")
    return ""


def get_assistant_fields(example: dict[str, Any]) -> tuple[str, str, dict[str, Any]]:
    """Return (think, tool_name, arguments)."""
    think = ""
    tool_name = example.get("tool_called", "none")
    arguments: dict[str, Any] = {}

    for message in example["messages"]:
        if message["role"] != "assistant":
            continue
        think = message.get("think") or message.get("_think_for_train") or ""
        tool_calls = message.get("tool_calls") or []
        if tool_calls:
            function = tool_calls[0]["function"]
            tool_name = function["name"]
            arguments = clean_arguments(function.get("arguments"))
        break

    return think, tool_name, arguments


# Try on the first example
think, tool_name, args = get_assistant_fields(example)
print("Think:", think[:120], "...")
print("Tool:", tool_name)
print("Args:", args)

### 4.2 Split prompt / completion

Trim at `<end_function_call>` because the dataset appends `<start_function_response>` after it (not part of training).

In [ ]:
def trim_completion(completion: str) -> str:
    if END_CALL in completion:
        end_index = completion.index(END_CALL) + len(END_CALL)
        return completion[:end_index]
    return completion.rstrip()


def split_prompt_and_completion(text: str) -> tuple[str, str]:
    if MODEL_TURN not in text:
        raise ValueError("Missing model turn marker in text field.")
    prompt, completion = text.split(MODEL_TURN, 1)
    prompt = prompt + MODEL_TURN
    return prompt, trim_completion(completion)


def to_track_a_completion(completion: str) -> str:
    """Track A: remove the <think> block."""
    return THINK_BLOCK_RE.sub("", completion).strip()


prompt, completion_b = split_prompt_and_completion(example["text"])
completion_a = to_track_a_completion(completion_b)

print("Prompt length:", len(prompt))
print("Track B completion length:", len(completion_b))
print("Track A completion length:", len(completion_a))
print("\n--- Track B completion ---")
print(completion_b)
print("\n--- Track A completion ---")
print(completion_a)

## 5. Build JSONL Records

Each line in a JSONL file = one training or evaluation example.

In [ ]:
def format_record(example: dict[str, Any], example_id: int, track: str) -> dict[str, Any]:
    prompt, completion_b = split_prompt_and_completion(example["text"])
    completion = completion_b if track.upper() == "B" else to_track_a_completion(completion_b)

    think, tool_name, arguments = get_assistant_fields(example)
    tools_sampled = [tool["function"]["name"] for tool in example.get("tools_sampled", [])]

    return {
        "id": example_id,
        "track": track.upper(),
        "prompt": prompt,
        "completion": completion,
        "text": prompt + completion,
        "tool_called": tool_name,
        "arguments": arguments,
        "think": think,
        "requires_function": example.get("requires_function", tool_name != "none"),
        "dialect": example.get("dialect"),
        "user": get_user_message(example),
        "tools_sampled": tools_sampled,
        "negative_category": example.get("negative_category"),
    }


def format_eval_record(example: dict[str, Any], example_id: int) -> dict[str, Any]:
    """For evaluation: prompt only + gold labels (no completion)."""
    prompt, _ = split_prompt_and_completion(example["text"])
    think, tool_name, arguments = get_assistant_fields(example)
    tools_sampled = [tool["function"]["name"] for tool in example.get("tools_sampled", [])]

    return {
        "id": example_id,
        "prompt": prompt,
        "user": get_user_message(example),
        "tool_called": tool_name,
        "arguments": arguments,
        "think": think,
        "requires_function": example.get("requires_function", tool_name != "none"),
        "dialect": example.get("dialect"),
        "tools_sampled": tools_sampled,
        "negative_category": example.get("negative_category"),
    }


# Preview one record
sample_b = format_record(example, 0, "B")
print(json.dumps({k: sample_b[k] for k in ["id", "track", "tool_called", "dialect", "user", "arguments"]}, ensure_ascii=False, indent=2))

## 6. Convert All Data

Apply the formatting functions to the full train and dev splits.

In [ ]:
train_track_b = [format_record(ex, i, "B") for i, ex in enumerate(train_ds)]
train_track_a = [format_record(ex, i, "A") for i, ex in enumerate(train_ds)]
dev_eval = [format_eval_record(ex, i) for i, ex in enumerate(dev_ds)]

print(f"Track B train: {len(train_track_b)}")
print(f"Track A train: {len(train_track_a)}")
print(f"Dev eval:      {len(dev_eval)}")

## 7. Validation

Verify the format is correct before saving.

In [ ]:
def validate_records(records: list[dict[str, Any]], track: str) -> None:
    for record in records:
        if not record["prompt"].endswith(MODEL_TURN):
            raise ValueError(f"Prompt missing model turn for id={record['id']}")
        if track.upper() == "A" and "<think>" in record["completion"]:
            raise ValueError(f"Track A still has think block for id={record['id']}")
        if track.upper() == "B" and record["requires_function"]:
            if "<start_function_call>" not in record["completion"]:
                raise ValueError(f"Track B missing function call for id={record['id']}")


validate_records(train_track_b, "B")
validate_records(train_track_a, "A")
print("Validation passed ✓")

## 8. Save JSONL Files

In [ ]:
def write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_stats(records: list[dict[str, Any]]) -> dict[str, Any]:
    return {
        "num_examples": len(records),
        "tool_counts": dict(Counter(r["tool_called"] for r in records)),
        "dialect_counts": dict(Counter(r["dialect"] for r in records)),
    }


write_jsonl(OUTPUT_DIR / "islamic_train_track_b.jsonl", train_track_b)
write_jsonl(OUTPUT_DIR / "islamic_train_track_a.jsonl", train_track_a)
write_jsonl(OUTPUT_DIR / "islamic_dev_eval.jsonl", dev_eval)

stats = {
    "train_track_b": build_stats(train_track_b),
    "train_track_a": build_stats(train_track_a),
    "dev_eval": build_stats(dev_eval),
    "notes": [
        "Track B includes <think> before the function call.",
        "Track A keeps only the Gemma function-call block.",
        "calculate_inheritance has 0 gold examples in the source dataset.",
    ],
}

with (OUTPUT_DIR / "stats.json").open("w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

preview = {
    "track_b": train_track_b[:PREVIEW_COUNT],
    "track_a": train_track_a[:PREVIEW_COUNT],
    "dev_eval": dev_eval[:PREVIEW_COUNT],
}
with (OUTPUT_DIR / "samples_preview.json").open("w", encoding="utf-8") as f:
    json.dump(preview, f, ensure_ascii=False, indent=2)

print(f"Saved to: {OUTPUT_DIR}")
print("Train tool counts:", stats["train_track_b"]["tool_counts"])

## 9. Preview Results

Quick review before handing off to Member 3 (training) and Member 4 (evaluation).

In [ ]:
import pandas as pd

rows = []
for rec in train_track_b[:10]:
    rows.append({
        "dialect": rec["dialect"],
        "tool": rec["tool_called"],
        "user": rec["user"][:50] + "..." if len(rec["user"]) > 50 else rec["user"],
        "args": str(rec["arguments"]),
    })

pd.DataFrame(rows)

## 10. Handoff Summary

### Member 3 — Fine-tuning
- Use `formatted/islamic_train_track_b.jsonl`
- Field `text` = prompt + completion (ready for causal LM SFT)
- Suggested base model: `google/gemma-3-270m` or `TuwaiqAcademy/AISA-AR-FunctionCall-Think`

### Member 4 — Evaluation
- Use `formatted/islamic_dev_eval.jsonl`
- Compare model output against gold `tool_called`, `arguments`, and `think`

> The same logic is available as a CLI script in `format_for_sft.py`.